In [1]:
# Cell 1: Environment Setup, Hyperparameters, and Data Fetching
import torch
import torch.nn as nn
from torch.nn import functional as F
import urllib.request
import os
import numpy as np
import matplotlib.pyplot as plt

# --- Hyperparameters ---
# Training & Data
batch_size = 32      # Number of independent sequences to process in parallel
block_size = 64      # Maximum context length for predictions (sequence length)
max_iters = 2500     # Total training steps
eval_interval = 250  # How often to evaluate the loss on train/val sets
eval_iters = 100     # Number of batches to average for evaluation
learning_rate = 1e-3 # Learning rate for the AdamW optimizer

# Model Architecture (Strictly following syllabus constraints)
n_embd = 128         # Hidden size (Requirement: <= 128)
n_head = 4           # Number of attention heads (128 / 4 = 32 dimensions per head)
n_layer = 2          # Number of Transformer blocks (Requirement: max 2)
vocab_size = 500     # Max vocabulary size for our subword tokenizer (Requirement: <= 500)
dropout = 0.1        # Dropout rate to prevent overfitting

# System Configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"System: Using device '{device}'")

# Ensure Reproducibility (Syllabus Requirement)
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# --- Data Fetching ---
# Download the Tiny Shakespeare dataset directly from Andrej Karpathy's repository
data_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
data_path = "input.txt"

if not os.path.exists(data_path):
    print("Downloading Tiny Shakespeare dataset...")
    urllib.request.urlretrieve(data_url, data_path)
    print("Download complete.")
else:
    print("Dataset already exists locally.")
    
with open(data_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text):,}")

System: Using device 'cuda'
Dataset already exists locally.
Length of dataset in characters: 1,115,394


In [2]:
# Cell 2: Custom BPE Tokenizer Training
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

print(f"Training custom BPE tokenizer with max vocab_size = {vocab_size}...")

# Initialize a tokenizer with a BPE model and a designated unknown token
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# We use ByteLevel pre-tokenization which is standard for GPT-style models
# It ensures we don't get out-of-vocabulary characters easily
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

# Create a trainer with our strict vocabulary size limit
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"], vocab_size=vocab_size)

# Train the tokenizer on our input text file
tokenizer.train(files=[data_path], trainer=trainer)

# Save the tokenizer to disk (good practice for reproducibility)
tokenizer_path = "tiny_shakespeare_bpe.json"
tokenizer.save(tokenizer_path)

actual_vocab_size = tokenizer.get_vocab_size()
print(f"Tokenizer trained and saved to {tokenizer_path}.")
print(f"Actual vocabulary size: {actual_vocab_size} (Must be <= 500)")

# --- Test the Tokenizer ---
sample_text = "O Romeo, Romeo! wherefore art thou Romeo?"
encoded = tokenizer.encode(sample_text)

print("\n--- Tokenizer Test ---")
print(f"Original Text: '{sample_text}'")
print(f"Encoded tokens: {encoded.tokens}")
print(f"Encoded IDs: {encoded.ids}")
# ByteLevel decoder requires a slightly different decode call, but standard decode works fine for inspection
print(f"Decoded Text: '{tokenizer.decode(encoded.ids)}'")

# Patching the Tokenizer's Decoder
from tokenizers import decoders

# Tell the tokenizer to translate 'Ġ' back to spaces and 'Ċ' back to newlines
tokenizer.decoder = decoders.ByteLevel()

# Re-save the fixed tokenizer just in case
tokenizer.save("tiny_shakespeare_bpe.json")

# Now test the decode function again
test_tokens = tokenizer.encode("O Romeo, Romeo! ").ids
print("Fixed Decode Output:", tokenizer.decode(test_tokens))

Training custom BPE tokenizer with max vocab_size = 500...



Tokenizer trained and saved to tiny_shakespeare_bpe.json.
Actual vocabulary size: 500 (Must be <= 500)

--- Tokenizer Test ---
Original Text: 'O Romeo, Romeo! wherefore art thou Romeo?'
Encoded tokens: ['O', 'ĠR', 'ome', 'o', ',', 'ĠR', 'ome', 'o', '!', 'Ġwhe', 're', 'fore', 'Ġar', 't', 'Ġthou', 'ĠR', 'ome', 'o', '?']
Encoded IDs: [29, 239, 159, 55, 8, 239, 159, 55, 4, 276, 77, 361, 347, 60, 155, 239, 159, 55, 14]
Decoded Text: 'O ĠR ome o , ĠR ome o ! Ġwhe re fore Ġar t Ġthou ĠR ome o ?'
Fixed Decode Output: O Romeo, Romeo! 


In [3]:
# Cell 3: Dataset Preparation and Batch Generation

# 1. Encode the entire dataset using our trained tokenizer
print("Encoding the dataset...")
# We extract just the IDs from the tokenizer's output
encoded_data = tokenizer.encode(text).ids
data = torch.tensor(encoded_data, dtype=torch.long)

print(f"Total tokens in dataset: {len(data):,}")
print(f"Vocabulary size used: {tokenizer.get_vocab_size()}")

# 2. Train/Validation Split (80% / 20% per syllabus)
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

# 3. Batch Generation Function
def get_batch(split):
    # Select the correct data split
    data_split = train_data if split == 'train' else val_data
    
    # Generate random starting indices for the sequences in our batch
    # We subtract block_size to ensure we don't read out of bounds when grabbing the +1 target
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    
    # Input: first N tokens (from index i to i + block_size)
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    # Target: shifted by one position (from index i+1 to i+1 + block_size)
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    
    # Move the tensors to our designated device (GPU or CPU)
    x, y = x.to(device), y.to(device)
    
    return x, y

# --- Test the Data Loader ---
xb, yb = get_batch('train')
print("\n--- Data Loader Test ---")
print(f"Input batch shape (x): {xb.shape} -> (batch_size, block_size)")
print(f"Target batch shape (y): {yb.shape} -> (batch_size, block_size)")

# Show the shift logic for the first sequence in the batch
print("\nSequence Shift Example (First 5 tokens of Sequence 0):")
print(f"Input : {xb[0, :5].tolist()}")
print(f"Target: {yb[0, :5].tolist()}")

Encoding the dataset...
Total tokens in dataset: 517,182
Vocabulary size used: 500
Training tokens: 413,745
Validation tokens: 103,437

--- Data Loader Test ---
Input batch shape (x): torch.Size([32, 64]) -> (batch_size, block_size)
Target batch shape (y): torch.Size([32, 64]) -> (batch_size, block_size)

Sequence Shift Example (First 5 tokens of Sequence 0):
Input : [52, 14, 350, 133, 69]
Target: [14, 350, 133, 69, 176]


In [4]:
# Cell 4: Core Attention Module
import torch.nn as nn

class Head(nn.Module):
    """ One head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
        # Register the causal mask as a buffer so PyTorch doesn't treat it as a trainable parameter
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        
        # Compute attention scores ("affinities")
        # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5) # scaled dot-product attention
        
        # Apply the causal mask to prevent attending to future tokens
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        
        # Perform the weighted aggregation of the values
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention operating in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Concatenate the outputs from all heads along the channel dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # Apply the final linear projection and dropout
        out = self.dropout(self.proj(out))
        return out

# --- Test the Attention Module ---
print("Testing Multi-Head Attention...")
# head_size is embedding_dimension divided by number of heads (128 / 4 = 32)
mha = MultiHeadAttention(n_head, n_embd // n_head).to(device)

# Pass a dummy batch through the module to ensure no dimension mismatches
dummy_x = torch.randn(batch_size, block_size, n_embd).to(device)
out = mha(dummy_x)

print(f"Input shape: {dummy_x.shape}")
print(f"Output shape: {out.shape} (Should perfectly match input shape)")

Testing Multi-Head Attention...
Input shape: torch.Size([32, 64, 128])
Output shape: torch.Size([32, 64, 128]) (Should perfectly match input shape)


In [5]:
# Cell 5: Transformer Block Components
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    """ Root Mean Square Normalization (Syllabus Requirement) """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # The learnable scale parameter
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        # Calculate RMS: x * 1 / sqrt(mean(x^2) + eps)
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        # Apply normalization in float32 for numerical stability, then cast back
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

class FeedForward(nn.Module):
    """ A simple multi-layer perceptron for token-level computation """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # Expand the hidden dimension by 4x, a standard Transformer convention
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            # Project back down to the original embedding dimension
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        
        # We apply RMSNorm before the attention and before the feed-forward (Pre-Norm formulation)
        self.rmsnorm1 = RMSNorm(n_embd)
        self.rmsnorm2 = RMSNorm(n_embd)

    def forward(self, x):
        # Pre-normalization with RMSNorm, then Self-Attention, then Residual connection
        x = x + self.sa(self.rmsnorm1(x))
        # Pre-normalization with RMSNorm, then Feed-Forward, then Residual connection
        x = x + self.ffwd(self.rmsnorm2(x))
        return x

# --- Test the Transformer Block ---
print("Testing Transformer Block with RMSNorm...")
block = Block(n_embd, n_head).to(device)

# We reuse dimensions from earlier to test
dummy_x = torch.randn(batch_size, block_size, n_embd).to(device)
out = block(dummy_x)

print(f"Input shape: {dummy_x.shape}")
print(f"Output shape: {out.shape} (Should perfectly match input shape)")

Testing Transformer Block with RMSNorm...
Input shape: torch.Size([32, 64, 128])
Output shape: torch.Size([32, 64, 128]) (Should perfectly match input shape)


In [6]:
# Cell 6: The Tiny Transformer Language Model
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

class TinyTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        # Token Embeddings: translates integer token IDs into dense vectors
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        
        # --- Sinusoidal Positional Encoding ---
        # Create a matrix of shape (block_size, n_embd)
        pe = torch.zeros(block_size, n_embd)
        # Create a vector of positions (0 to block_size - 1)
        position = torch.arange(0, block_size, dtype=torch.float).unsqueeze(1)
        # Compute the frequency denominator: 10000^(2i/n_embd)
        div_term = torch.exp(torch.arange(0, n_embd, 2).float() * (-math.log(10000.0) / n_embd))
        
        # Apply sine to even indices, cosine to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Register as a buffer so PyTorch manages it, but it isn't a trainable parameter
        self.register_buffer('pe', pe)
        
        # --- Transformer Blocks ---
        # Stack n_layer (2) blocks sequentially
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        
        # --- Final Layers ---
        self.rmsnorm_f = RMSNorm(n_embd) # Final normalization before the classifier
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False) # Output projection

        # Weight Initialization (Standard practice for Transformers)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # 1. Embeddings
        tok_emb = self.token_embedding_table(idx) # (Batch, Time, Channels)
        pos_emb = self.pe[:T, :]                  # (Time, Channels)
        
        # Broadcasting adds pos_emb to every sequence in the batch
        x = tok_emb + pos_emb                     
        
        # 2. Transformer Blocks
        x = self.blocks(x)                        
        
        # 3. Final Norm and Output Logits
        x = self.rmsnorm_f(x)                     
        logits = self.lm_head(x)                  # (Batch, Time, Vocab_Size)
        
        # 4. Loss Calculation (Cross-Entropy for Next-Token Prediction)
        if targets is None:
            loss = None
        else:
            # PyTorch cross_entropy expects the channel (vocab) dimension to be the second dimension
            # Or we can flatten the Batch and Time dimensions
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)
            
        return logits, loss

# --- Instantiate the Model ---
model = TinyTransformer().to(device)

# --- Test the Full Model ---
print("Testing the Tiny Transformer Model...")
xb, yb = get_batch('train')
logits, loss = model(xb, yb)

print(f"Logits shape: {logits.shape} -> (batch_size, block_size, vocab_size)")
print(f"Initial untrained loss: {loss.item():.4f}")

# A perfectly random model should have a loss roughly equal to -ln(1/vocab_size)
expected_initial_loss = -math.log(1.0 / vocab_size)
print(f"Expected initial loss (approx): {expected_initial_loss:.4f}")

Testing the Tiny Transformer Model...
Logits shape: torch.Size([32, 64, 500]) -> (batch_size, block_size, vocab_size)
Initial untrained loss: 6.2192
Expected initial loss (approx): 6.2146


In [7]:
# Cell 7: Training Loop and Evaluation Logic

# Lists to store metrics for our final plots (Syllabus Requirement)
train_losses = []
val_losses = []
val_perplexities = []
track_iters = []

@torch.no_grad()
def estimate_loss():
    """ Evaluates the model on both train and val splits without tracking gradients """
    out = {}
    model.eval() # Set model to evaluation mode (disables dropout)
    
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
        
    model.train() # Set model back to training mode
    return out

In [8]:
# Cell 9: Text Generation

@torch.no_grad()
def generate_text(model, start_text, max_new_tokens=250):
    """
    Autoregressively generates new text given a starting prompt.
    """
    model.eval() # Ensure dropout is disabled for generation
    
    # Encode the starting text and move it to the device
    # Shape: (1, T) where 1 is the batch size
    context = torch.tensor(tokenizer.encode(start_text).ids, dtype=torch.long, device=device).unsqueeze(0)
    
    for _ in range(max_new_tokens):
        # Crop the context to the maximum block_size to prevent index out of bounds
        context_cropped = context[:, -block_size:]
        
        # Get the predictions
        logits, _ = model(context_cropped)
        
        # Focus only on the predictions for the very last time step
        logits = logits[:, -1, :] # Becomes (Batch, Vocab_Size)
        
        # Apply softmax to convert logits to probabilities
        probs = F.softmax(logits, dim=-1)
        
        # Sample the next token from the probability distribution
        next_token = torch.multinomial(probs, num_samples=1) # (Batch, 1)
        
        # Append the sampled token to the running sequence
        context = torch.cat((context, next_token), dim=1)
        
    # Decode the final sequence of integer IDs back into a string
    generated_ids = context[0].tolist()
    
    # Use tokenizer.decode to reconstruct the text
    # (Note: BPE decode automatically handles spacing and subword merging)
    return tokenizer.decode(generated_ids)

In [9]:
# Cell 10: Automated Experimentation Pipeline

def run_experiment(exp_name, custom_params):
    """
    Runs a full training and evaluation pipeline with new hyperparameters.
    """
    print(f"\n{'='*60}")
    print(f"🚀 STARTING EXPERIMENT: {exp_name}")
    print(f"Parameters: {custom_params}")
    print(f"{'='*60}")
    
    # 1. Update the global variables so Cells 1-9 use the new settings
    globals().update(custom_params)
    
    # 2. Instantiate a fresh model and optimizer with the new globals
    global model # Ensure the global model variable is updated for the estimate_loss function
    model = TinyTransformer().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    
    # Local tracking lists for this specific experiment
    exp_train_losses = []
    exp_val_losses = []
    exp_track_iters = []
    
    # 3. Training Loop
    for iter in range(max_iters):
        if iter % eval_interval == 0 or iter == max_iters - 1:
            losses = estimate_loss() # Uses the global estimate_loss function
            exp_train_losses.append(losses['train'])
            exp_val_losses.append(losses['val'])
            exp_track_iters.append(iter)
            print(f"Step {iter:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

        xb, yb = get_batch('train')
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        
    print(f"✅ Training Complete. Saving artifacts for {exp_name}...")
    
    # 4. Save Loss Curves
    plt.figure(figsize=(10, 4))
    plt.plot(exp_track_iters, exp_train_losses, label='Train Loss', color='blue')
    plt.plot(exp_track_iters, exp_val_losses, label='Validation Loss', color='orange')
    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.title(f'Loss Curves: {exp_name}')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(f'{exp_name}_loss.png', bbox_inches='tight', dpi=300)
    plt.close() # Close plot so it doesn't spam the notebook UI
    
    # 5. Save Attention Heatmap (Custom function to save with exp_name)
    save_experiment_attention_map(exp_name, "O Romeo, Romeo! wherefore art thou")
    
    # 6. Save Generated Text
    start_prompt = "O Romeo, Romeo! "
    sample_text = generate_text(model, start_prompt, max_new_tokens=200)
    with open(f'{exp_name}_sample.txt', 'w', encoding='utf-8') as f:
        # Include the params at the top of the text file for easy reference
        f.write(f"Experiment: {exp_name}\nParams: {custom_params}\n{'-'*40}\n")
        f.write(sample_text)
        
    # NEW: Get peak VRAM usage in MB
    peak_vram = torch.cuda.max_memory_allocated(device) / 1024**2
    print(f"📈 Peak VRAM Usage: {peak_vram:.2f} MB")
    
    # Reset the peak counter so the next experiment starts fresh
    torch.cuda.reset_peak_memory_stats(device)
    # 7. Save Model Weights
    torch.save(model.state_dict(), f'{exp_name}_weights.pth')
    print(f"💾 All files saved with prefix: {exp_name}_")

@torch.no_grad()
def save_experiment_attention_map(exp_name, text_snippet):
    """ Helper to save attention map with the experiment name """
    model.eval()
    tokens = tokenizer.encode(text_snippet).ids
    x = torch.tensor([tokens]).to(device)
    T = x.size(1)
    
    head = model.blocks[0].sa.heads[0]
    tok_emb = model.token_embedding_table(x)
    pos_emb = model.pe[:T, :]
    emb = tok_emb + pos_emb
    norm_x = model.blocks[0].rmsnorm1(emb)
    q = head.query(norm_x)
    k = head.key(norm_x)
    wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
    tril = torch.tril(torch.ones(T, T)).to(device)
    wei = wei.masked_fill(tril == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    
    attn_matrix = wei[0].cpu().numpy()
    token_labels = [tokenizer.decode([t]).strip() or repr(tokenizer.decode([t])) for t in tokens]
    
    plt.figure(figsize=(8, 8))
    plt.imshow(attn_matrix, cmap='viridis')
    plt.xticks(range(T), token_labels, rotation=45, ha='right')
    plt.yticks(range(T), token_labels)
    plt.title(f"Causal Attention: {exp_name}")
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(f'{exp_name}_attention.png', bbox_inches='tight', dpi=300)
    plt.close()

In [10]:
# Cell 11: Execute Hyperparameter Sweep

# Naming convention: [Description]_[Embedding Dimension]d_[Attention Heads]h_[Layers]L
experiments = [
    {
        "name": "exp1_tiny_baseline_64d_4h_2L",
        "params": {"n_embd": 64, "n_head": 4, "n_layer": 2, "max_iters": 2000}
    },
    {
        "name": "exp2_64d_4h_4L",
        "params": {"n_embd": 64, "n_head": 4, "n_layer": 4, "max_iters": 2000}
    },
    {
        "name": "exp3_128d_4h_1L",
        "params": {"n_embd": 128, "n_head": 4, "n_layer": 1, "max_iters": 2000}
    },
    {
        "name": "exp4_512d_8h_16L",
        "params": {"n_embd": 512, "n_head": 8, "n_layer": 16, "max_iters": 2000}
    },
    {
        "name": "exp5_512d_16h_16L",
        "params": {"n_embd": 512, "n_head": 16, "n_layer": 16, "max_iters": 2000}
    },
    {
        "name": "exp6_512d_8h_32L",
        "params": {"n_embd": 512, "n_head": 8, "n_layer": 32, "max_iters": 2000}
    },
    {
        "name": "exp7_extreme_512d_32h_128L",
        "params": {"n_embd": 512, "n_head": 32, "n_layer": 128, "max_iters": 2000}
    }
]

# Run them sequentially
for exp in experiments:
    print(f"🚀 Starting {exp['name']}...")
    run_experiment(exp["name"], exp["params"])
    
print("\n🎉 ALL EXPERIMENTS COMPLETED!")

🚀 Starting exp1_tiny_baseline_64d_4h_2L...

🚀 STARTING EXPERIMENT: exp1_tiny_baseline_64d_4h_2L
Parameters: {'n_embd': 64, 'n_head': 4, 'n_layer': 2, 'max_iters': 2000}
Step    0 | Train Loss: 6.2341 | Val Loss: 6.2332
Step  250 | Train Loss: 4.6479 | Val Loss: 4.6890
Step  500 | Train Loss: 3.9024 | Val Loss: 4.1006
Step  750 | Train Loss: 3.6568 | Val Loss: 3.9198
Step 1000 | Train Loss: 3.5445 | Val Loss: 3.8295
Step 1250 | Train Loss: 3.4435 | Val Loss: 3.7624
Step 1500 | Train Loss: 3.3775 | Val Loss: 3.7464
Step 1750 | Train Loss: 3.3208 | Val Loss: 3.6854
Step 1999 | Train Loss: 3.2782 | Val Loss: 3.7020
✅ Training Complete. Saving artifacts for exp1_tiny_baseline_64d_4h_2L...
📈 Peak VRAM Usage: 132.59 MB
💾 All files saved with prefix: exp1_tiny_baseline_64d_4h_2L_
🚀 Starting exp2_64d_4h_4L...

🚀 STARTING EXPERIMENT: exp2_64d_4h_4L
Parameters: {'n_embd': 64, 'n_head': 4, 'n_layer': 4, 'max_iters': 2000}
Step    0 | Train Loss: 6.2290 | Val Loss: 6.2277
Step  250 | Train Loss: 4.